In [2]:
import os
import pandas as pd

from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

# Credenciales de las bases de datos de origen

DB_USER = os.getenv("POSTGRES_USER", "admin")
DB_PASSWORD = os.getenv("POSTGRES_PASSWORD")

# Nombre de la base de datos y servicio Docker para cada fuente

print("USER und PASSWORT;" + DB_USER + DB_PASSWORD )

DATABASES = {
"City Move": {
"host": "city_move",
"database": os.getenv("CITYMOVE_DB", "city_move"),
},
"Bike Share": {
"host": "bikeshare",
"database": os.getenv("BIKESHARE_DB", "bike_share"),
},
"Airsense": {
"host": "airsense",
"database": os.getenv("AIRSENSE_DB", "airsense"),
},
"City Events": {
"host": "cityevents",
"database": os.getenv("CITYEVENTS_DB", "city_events"),
},
}

def create_source_engine(host, database):
    url = URL.create(
    drivername="postgresql+psycopg",
    username=DB_USER,
    password=DB_PASSWORD,
    host=host,
    port=5432,
    database=database,
    )
    return create_engine(url)
    
print("Sources Configured:")
for name, config in DATABASES.items():
    print(f"- {name}: {config['host']} / {config['database']}")


USER und PASSWORT;adminadmin123
Sources Configured:
- City Move: city_move / city_move
- Bike Share: bikeshare / bike_share
- Airsense: airsense / airsense
- City Events: cityevents / city_events


In [3]:
for name, config in DATABASES.items():
    engine = create_source_engine(
    config["host"],
    config["database"],
    )

    try:
        with engine.connect() as connection:
            result = connection.execute(
                text("""
                    SELECT
                        current_database() AS database_name,
                        current_user AS user_name;
                """)
            )
            row = result.fetchone()
    
        print(
            f"OK | {name} | "
            f"DB: {row.database_name} | "
            f"USER: {row.user_name}"
        )
    
    except Exception as exc:
        print(f"ERROR | {name} | {type(exc).__name__}: {exc}")
    
    finally:
        engine.dispose()


OK | City Move | DB: city_move | USER: admin
OK | Bike Share | DB: bike_share | USER: admin
OK | Airsense | DB: airsense | USER: admin
OK | City Events | DB: city_events | USER: admin


In [ ]:
# ------------------------------------------------------------------

In [4]:
from sqlalchemy import text
import pandas as pd

def explore_database(name, config):
    engine = create_source_engine(
        config["host"],
        config["database"]
    )

    try:
        # Tabellen abrufen
        query_tables = text("""
            SELECT table_schema, table_name
            FROM information_schema.tables
            WHERE table_type = 'BASE TABLE'
              AND table_schema NOT IN (
                  'pg_catalog',
                  'information_schema'
              )
            ORDER BY table_schema, table_name;
        """)

        with engine.connect() as connection:
            tables = pd.read_sql(query_tables, connection)

        if tables.empty:
            print(f"keine Tabelle gefunden :(  {name}.")
            return pd.DataFrame()

        results = []

        # Anzahl der Zeilen pro Tabelle
        with engine.connect() as connection:
            for _, row in tables.iterrows():
                schema = row["table_schema"]
                table = row["table_name"]

                quoted_schema = '"' + schema.replace('"', '""') + '"'
                quoted_table = '"' + table.replace('"', '""') + '"'

                count_query = text(
                    f"SELECT COUNT(*) FROM {quoted_schema}.{quoted_table}"
                )

                count = connection.execute(count_query).scalar_one()

                results.append({
                    "database": name,
                    "schema": schema,
                    "table": table,
                    "rows": count
                })

        return pd.DataFrame(results)

    finally:
        engine.dispose()


# -----------------------------
# Hauptlogik
# -----------------------------

all_results = []

for name, config in DATABASES.items():
    print(f"\nExplore {name}...")

    try:
        result = explore_database(name, config)

        if not result.empty:
            display(result)
            all_results.append(result)

    except Exception as exc:
        print(f"Error explorando {name}: {exc}")


if all_results:
    table_summary = pd.concat(all_results, ignore_index=True)

    print("\nZusammenfassung:")
    display(
        table_summary.sort_values(
            ["database", "rows"],
            ascending=[True, False]
        ).reset_index(drop=True)
    )




Explore City Move...


,database,schema,table,rows
0,City Move,citymove,fahrgastaufkommen,2065
1,City Move,citymove,fahrplan,2903
2,City Move,citymove,fahrt,300
3,City Move,citymove,fahrt_haltestelle,2065
4,City Move,citymove,fahrzeug,50
5,City Move,citymove,haltestelle,40
6,City Move,citymove,linie,10
7,City Move,citymove,linien_haltestelle,190
8,City Move,citymove,modell,8
9,City Move,citymove,stoerung,30



Explore Bike Share...


,database,schema,table,rows
0,Bike Share,bikeshare,fahrrad,150
1,Bike Share,bikeshare,fahrradmodell,8
2,Bike Share,bikeshare,fahrt,1000
3,Bike Share,bikeshare,kunde,300
4,Bike Share,bikeshare,station,24
5,Bike Share,bikeshare,tarif,5
6,Bike Share,bikeshare,wartung,205
7,Bike Share,bikeshare,zahlung,782



Explore Airsense...


,database,schema,table,rows
0,Airsense,airsense,messung,1000
1,Airsense,airsense,sensor,88
2,Airsense,airsense,sensorwartung,133
3,Airsense,airsense,station,25



Explore City Events...


,database,schema,table,rows
0,City Events,cityevents,besucher,190
1,City Events,cityevents,einlass,335
2,City Events,cityevents,stoerung,12
3,City Events,cityevents,ticket,1000
4,City Events,cityevents,veranstalter,12
5,City Events,cityevents,veranstaltung,40
6,City Events,cityevents,veranstaltungsort,18
7,City Events,cityevents,veranstaltungstermin,80
8,City Events,cityevents,veranstaltungstyp,10



Zusammenfassung:


,database,schema,table,rows
0,Airsense,airsense,messung,1000
1,Airsense,airsense,sensorwartung,133
2,Airsense,airsense,sensor,88
3,Airsense,airsense,station,25
4,Bike Share,bikeshare,fahrt,1000
5,Bike Share,bikeshare,zahlung,782
6,Bike Share,bikeshare,kunde,300
7,Bike Share,bikeshare,wartung,205
8,Bike Share,bikeshare,fahrrad,150
9,Bike Share,bikeshare,station,24


In [5]:
from sqlalchemy import text
import pandas as pd

config = DATABASES["City Move"]

engine = create_source_engine(
    config["host"],
    config["database"]
)

tables_to_inspect = [
    "fahrt",
    "fahrplan",
    "fahrt_haltestelle",
    "fahrgastaufkommen",
    "stoerung"
]

for table in tables_to_inspect:
    print(f"\n### {table}")

    query = text("""
        SELECT
            column_name,
            data_type,
            is_nullable
        FROM information_schema.columns
        WHERE table_schema = 'citymove'
          AND table_name = :table_name
        ORDER BY ordinal_position;
    """)

    with engine.connect() as connection:
        columns = pd.read_sql(
            query,
            connection,
            params={"table_name": table}
        )

    display(columns)


### fahrt


,column_name,data_type,is_nullable
0,id,integer,NO
1,id_linie,integer,YES
2,id_fahrzeug,integer,YES
3,datum,date,YES
4,fahrtrichtung,character varying,YES
5,fahrten_status,character varying,YES



### fahrplan


,column_name,data_type,is_nullable
0,id,integer,NO
1,id_fahrt,integer,YES
2,id_haltestelle,integer,YES
3,ankunft_geplant,timestamp without time zone,YES
4,abfahrt_geplant,timestamp without time zone,YES



### fahrt_haltestelle


,column_name,data_type,is_nullable
0,id,integer,NO
1,id_fahrt,integer,YES
2,id_haltestelle,integer,YES
3,ankunft_tatsaechlich,timestamp without time zone,YES
4,abfahrt_tatsaechlich,timestamp without time zone,YES



### fahrgastaufkommen


,column_name,data_type,is_nullable
0,id,integer,NO
1,id_fahrt,integer,YES
2,id_haltestelle,integer,YES
3,einsteiger,integer,YES
4,aussteiger,integer,YES



### stoerung


,column_name,data_type,is_nullable
0,id,integer,NO
1,id_fahrt,integer,YES
2,id_haltestelle,integer,YES
3,typ,character varying,YES
4,beginn,timestamp without time zone,YES
5,ende,timestamp without time zone,YES
6,dauer_min,integer,YES
7,beschreibung,character varying,YES


In [6]:
query = text("""
    SELECT
        f.id AS id_fahrt,
        f.id_linie,
        f.id_fahrzeug,
        f.datum,
        f.fahrtrichtung,

        fp.id_haltestelle,
        fp.ankunft_geplant,

        fh.ankunft_tatsaechlich,

        EXTRACT(
            EPOCH FROM (
                fh.ankunft_tatsaechlich
                - fp.ankunft_geplant
            )
        ) / 60.0 AS verspaetung_minuten

    FROM citymove.fahrplan fp

    INNER JOIN citymove.fahrt_haltestelle fh
        ON fp.id_fahrt = fh.id_fahrt
       AND fp.id_haltestelle = fh.id_haltestelle

    INNER JOIN citymove.fahrt f
        ON f.id = fp.id_fahrt

    WHERE fp.ankunft_geplant IS NOT NULL
      AND fh.ankunft_tatsaechlich IS NOT NULL

    ORDER BY f.datum, f.id, fp.id_haltestelle

    LIMIT 20;
""")

with engine.connect() as connection:
    sample = pd.read_sql(query, connection)

display(sample)

,id_fahrt,id_linie,id_fahrzeug,datum,fahrtrichtung,id_haltestelle,ankunft_geplant,ankunft_tatsaechlich,verspaetung_minuten
0,120,2,41,2026-03-31,Rückfahrt,2,2026-03-31 20:54:00,2026-03-31 20:57:00,3.0
1,120,2,41,2026-03-31,Rückfahrt,3,2026-03-31 20:57:00,2026-03-31 21:05:00,8.0
2,120,2,41,2026-03-31,Rückfahrt,4,2026-03-31 21:27:00,2026-03-31 21:32:00,5.0
3,120,2,41,2026-03-31,Rückfahrt,12,2026-03-31 20:50:00,2026-03-31 20:52:00,2.0
4,120,2,41,2026-03-31,Rückfahrt,25,2026-03-31 20:52:00,2026-03-31 20:57:00,5.0
5,120,2,41,2026-03-31,Rückfahrt,26,2026-03-31 21:02:00,2026-03-31 21:10:00,8.0
6,120,2,41,2026-03-31,Rückfahrt,33,2026-03-31 21:06:00,2026-03-31 21:08:00,2.0
7,120,2,41,2026-03-31,Rückfahrt,35,2026-03-31 21:23:00,2026-03-31 21:25:00,2.0
8,120,2,41,2026-03-31,Rückfahrt,38,2026-03-31 21:12:00,2026-03-31 21:13:00,1.0
9,120,2,41,2026-03-31,Rückfahrt,39,2026-03-31 21:17:00,2026-03-31 21:29:00,12.0
